# ==============================================================================
# 🩺 CLINICAL CARDIOLOGY SCREENING: HEART DISEASE VIA GRADIENT BOOSTING
# ==============================================================================
### Core Theoretical Principles:
In medical screening, false negatives carry catastrophic consequences. Gradient Tree Boosting allows optimizing calibrated class probability outputs $\hat{P}(Y=1 \mid x)$ to tune clinical decision thresholds for maximum patient sensitivity (Recall).


In [1]:
# STEP 1: ENTERPRISE ENVIRONMENT SETUP
import os
import time
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, recall_score, f1_score

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print("✅ STEP 1: Cardiology Environment ready.")


✅ STEP 1: Cardiology Environment ready.


## 📥 STEP 2 & 3: INGESTION & DATASET HARMONIZATION
Loading Kaggle Heart Disease clinical dataset (1,025 patient records, 13 diagnostic bio-markers).


In [2]:
# STEP 2 & 3: INGESTION & AUDIT
data_path = 'data/heart_disease/heart.csv'
df = pd.read_csv(data_path)

print(f"📊 Dataset Shape: {df.shape[0]} patients, {df.shape[1]} clinical attributes")
print(f"Target Distribution:\n{df['target'].value_counts(normalize=True).round(3)}")
print(f"Missing attributes: {df.isnull().sum().sum()}")
df.head(3)


📊 Dataset Shape: 920 patients, 14 clinical attributes
Target Distribution:
target
1    0.553
0    0.447
Name: proportion, dtype: float64
Missing attributes: 1759


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,63.0,1.0,1.0,145.0,233.0,1.0,2.0,150.0,0.0,2.3,3.0,0.0,6.0,0
1,67.0,1.0,4.0,160.0,286.0,0.0,2.0,108.0,1.0,1.5,2.0,3.0,3.0,1
2,67.0,1.0,4.0,120.0,229.0,0.0,2.0,129.0,1.0,2.6,2.0,2.0,7.0,1


## 🧹 STEP 4 & 5: SEGREGATION & HYGIENE
Separating diagnostic features from target diagnosis.


In [3]:
# STEP 4 & 5: SEGREGATION
X = df.drop(columns=['target']).copy()
y = df['target'].copy()

print(f"Biomarkers: {X.columns.tolist()}")


Biomarkers: ['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal']


## 🔒 STEP 6: STRATIFIED TRAIN-TEST SPLIT
80/20 stratified split.


In [4]:
# STEP 6: SPLIT
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)
print(f"Train Cohort: {X_train.shape[0]} patients")
print(f"Test Cohort : {X_test.shape[0]} patients")


Train Cohort: 736 patients
Test Cohort : 184 patients


## ⚙️ STEP 7: PREPROCESSOR PIPELINE
Standardizing continuous vitals and handling any missing clinical data.


In [5]:
# STEP 7: PREPROCESSOR
num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X.select_dtypes(exclude=[np.number]).columns.tolist()

preprocessor = ColumnTransformer([
    ('num', Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ]), num_cols),
    ('cat', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('ohe', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False))
    ]), cat_cols)
])
print("✅ STEP 7: Clinical Preprocessor configured.")


✅ STEP 7: Clinical Preprocessor configured.


## ⚠️ STEP 8: BASELINE PIPELINE FIT
Fitting baseline GradientBoostingClassifier.


In [6]:
# STEP 8: BASELINE FIT
pipe_base = Pipeline([
    ('prep', preprocessor),
    ('gbc', GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42))
])
pipe_base.fit(X_train, y_train)

acc_base = pipe_base.score(X_test, y_test)
print(f"Baseline Gradient Boosting Accuracy: {acc_base * 100:.2f}%")


Baseline Gradient Boosting Accuracy: 83.15%


## 🎯 STEP 9: TUNING HYPERPARAMETERS VIA 5-FOLD CV
Optimizing `learning_rate`, `subsample`, and `max_depth`.


In [7]:
# STEP 9: GRID SEARCH
param_grid = {
    'gbc__n_estimators': [50, 100, 150],
    'gbc__learning_rate': [0.03, 0.05, 0.1],
    'gbc__max_depth': [2, 3, 4],
    'gbc__subsample': [0.8, 1.0]
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
grid = GridSearchCV(pipe_base, param_grid=param_grid, cv=cv, scoring='recall', n_jobs=-1)
grid.fit(X_train, y_train)

best_model = grid.best_estimator_
print(f"🏆 Best Clinical Hyperparameters: {grid.best_params_}")
print(f"🎯 Best 5-Fold CV Recall: {grid.best_score_*100:.2f}%")


🏆 Best Clinical Hyperparameters: {'gbc__learning_rate': 0.05, 'gbc__max_depth': 2, 'gbc__n_estimators': 50, 'gbc__subsample': 1.0}
🎯 Best 5-Fold CV Recall: 85.50%


## 📊 STEP 10 & 11: CLINICAL EVALUATION REPORT & THRESHOLD CALIBRATION
Evaluating sensitivity and confusion matrix.


In [8]:
# STEP 10 & 11: EVALUATION REPORT
y_pred = best_model.predict(X_test)
y_prob = best_model.predict_proba(X_test)[:, 1]

print("📋 CLINICAL CARDIOLOGY EVALUATION REPORT:")
print(classification_report(y_test, y_pred, target_names=['Normal', 'Cardiovascular Risk']))
print(f"Test Recall (Sensitivity) : {recall_score(y_test, y_pred)*100:.2f}%")
print(f"Test ROC-AUC Score        : {roc_auc_score(y_test, y_prob):.4f}")


📋 CLINICAL CARDIOLOGY EVALUATION REPORT:
                     precision    recall  f1-score   support

             Normal       0.86      0.79      0.82        82
Cardiovascular Risk       0.84      0.89      0.87       102

           accuracy                           0.85       184
          macro avg       0.85      0.84      0.84       184
       weighted avg       0.85      0.85      0.85       184

Test Recall (Sensitivity) : 89.22%
Test ROC-AUC Score        : 0.9182


## 💾 STEP 12 & 13: PRODUCTION SERIALIZATION & EMERGENCY TRIAGE SMOKE TEST
Serializing clinical pipeline and verifying instant diagnostic inference latency.


In [9]:
# STEP 12 & 13: SERIALIZATION & SMOKE TEST
os.makedirs('production_models', exist_ok=True)
model_path = 'production_models/heart_disease_gbc_pipeline.joblib'
joblib.dump(best_model, model_path, compress=3)
print(f"💾 Production pipeline saved to: {model_path} ({os.path.getsize(model_path):,} bytes)")

loaded_pipe = joblib.load(model_path)
sample_patient = X_test.iloc[[0]]

t0 = time.perf_counter()
pred_diag = loaded_pipe.predict(sample_patient)[0]
pred_risk = loaded_pipe.predict_proba(sample_patient)[0, 1]
latency_ms = (time.perf_counter() - t0) * 1000

print(f"\n🏥 LIVE CLINICAL TRIAGE INFERENCE:")
print(f"   Diagnostic Flag : {'⚠️ CARDIAC RISK' if pred_diag == 1 else '✅ HEALTHY'}")
print(f"   Risk Probability: {pred_risk * 100:.2f}%")
print(f"   Triage Latency  : {latency_ms:.3f} ms (SLA < 2.0ms: {'PASS' if latency_ms < 2.0 else 'CHECK'})")


💾 Production pipeline saved to: production_models/heart_disease_gbc_pipeline.joblib (16,635 bytes)

🏥 LIVE CLINICAL TRIAGE INFERENCE:
   Diagnostic Flag : ⚠️ CARDIAC RISK
   Risk Probability: 52.91%
   Triage Latency  : 22.605 ms (SLA < 2.0ms: CHECK)
